# BAC token-bottleneck testbed (Kaggle, GPU T4 x2)
The M compressed tokens are the only route from SegEarth-R1's frozen Swin-B features to the mask. 64-token comparison (pool, strided conv, BAC, BAC-v2, BAC-v2 with random selection) x 3 seeds, plus the uncompressed upper bound (seeds 1-2). The 16-token comparison runs on the Mac.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import sys, torch; print(sys.version.split()[0], '| torch', torch.__version__, '| CUDA GPUs:', torch.cuda.device_count())" 

In [ ]:
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'    # PyTorch only (Kaggle's JAX needs NumPy 2; we keep Kaggle's NumPy)
!pip -q install transformers==4.36.2 tokenizers==0.15.2 timm==0.6.13 fvcore addict einops einops-exts==0.0.4 accelerate==0.21.0 peft==0.4.0 tabulate 2>&1 | tail -2
!python -c "import numpy, transformers, timm, torch; print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| timm', timm.__version__, '| torch', torch.__version__)" 

In [ ]:
import glob, os, shutil, zipfile
# Code, weights, data and the 5.6 GB feature cache live in /kaggle/tmp (not saved). Only results/ is saved:
# it is a link into /kaggle/working, so even a crashed run leaves a small, downloadable output.
W = '/kaggle/tmp/Minor'; os.makedirs(W, exist_ok=True)
zips = glob.glob('/kaggle/input/**/bac_code.zip', recursive=True)
if zips:                                    # dataset kept the zip
    zipfile.ZipFile(zips[0]).extractall(W)
else:                                       # Kaggle auto-extracted the zip: copy the extracted tree
    marker = glob.glob('/kaggle/input/**/bac/common.py', recursive=True)
    assert marker, 'Add the bac-code dataset via Add Input'
    shutil.copytree(os.path.dirname(os.path.dirname(marker[0])), W, dirs_exist_ok=True)
os.makedirs('/kaggle/working/results/logs', exist_ok=True)
if not os.path.islink(f'{W}/results'):
    shutil.rmtree(f'{W}/results', ignore_errors=True); os.symlink('/kaggle/working/results', f'{W}/results')
os.chdir(W)
!ls; df -h /kaggle/working /kaggle/tmp | cat

In [ ]:
# downloads run in a subprocess so the pinned libraries are used everywhere
!python -c "from huggingface_hub import snapshot_download; snapshot_download('earth-insights/SegEarth-R1-EarthReason', local_dir='checkpoints/SegEarth-R1-EarthReason')"
!python -c "from huggingface_hub import hf_hub_download; import zipfile; [zipfile.ZipFile(hf_hub_download('earth-insights/EarthReason', f'{s}.zip', repo_type='dataset', local_dir='/kaggle/tmp')).extractall('data/rs_reason_seg/RSReasonSeg') for s in ('val', 'train')]"
!ls data/rs_reason_seg/RSReasonSeg/train/images | wc -l; ls data/rs_reason_seg/RSReasonSeg/val/images | wc -l; du -sh checkpoints

In [ ]:
# fail fast: if the environment is broken, stop here instead of skipping through hours of runs
import subprocess, json, os, sys
r = subprocess.run('python bac/evaluate.py --name smoke --limit 5 --out /kaggle/tmp/smoke', shell=True, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:])
assert os.path.exists('/kaggle/tmp/smoke/smoke_val.json'), 'SMOKE TEST FAILED: evaluation does not run - see error above'
print('smoke eval OK, gIoU on 5 images:', round(json.load(open('/kaggle/tmp/smoke/smoke_val.json'))['gIoU'], 3))

In [ ]:
r = subprocess.run('python bac/train.py --name smoke --kind bac --tokens 64 --steps 3', shell=True, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-3000:])
assert os.path.exists('runs/smoke/connector.pt'), 'SMOKE TEST FAILED: training does not run - see error above'
import shutil; shutil.rmtree('runs/smoke'); print('smoke train OK')

In [ ]:
!python bac/test_connectors.py 2>&1 | grep -E "OK|passed|Error"

In [ ]:
# testbed smoke test: every compressor family through the exact queue path, on 12 images, before the 30-min extraction
r = subprocess.run('python bac/token_bottleneck/extract_features.py --smoke 12', shell=True, capture_output=True, text=True)
print(r.stdout[-600:], r.stderr[-3000:])
bad = []
for n, a in {'pool': '--kind pool --tokens 64', 'uniform': '--kind uniform --tokens 64', 'bac': '--kind bac --tokens 64',
             'bac2': '--kind bac2 --tokens 64', 'bac2r': '--kind bac2 --tokens 16 --select random', 'bacq': '--kind bacq --tokens 16',
             'tome': '--kind tome --tokens 16', 'random': '--kind random --tokens 16', 'pool1024': '--kind pool --tokens 1024'}.items():
    r = subprocess.run(f'PY=python bash bac/token_bottleneck/run.sh smoke_{n} 1 0 {a} --feat-dir data/bottleneck_smoke --max-steps 10 --batch 4 --out-dir /kaggle/tmp/tbsmoke',
                       shell=True, capture_output=True, text=True)
    ok = os.path.exists(f'/kaggle/tmp/tbsmoke/smoke_{n}_s0.json'); print(n, 'OK' if ok else 'FAILED'); bad += [] if ok else [n]
    if not ok: print(r.stdout[-3000:], r.stderr[-2000:])
shutil.rmtree('data/bottleneck_smoke', ignore_errors=True); shutil.rmtree('results/logs/runs', ignore_errors=True)
assert not bad, f'TESTBED SMOKE FAILED: {bad}'

In [ ]:
!EPOCHS=6 bash kaggle/testbed_queue.sh

In [ ]:
import json, glob, pandas as pd
rows = [json.load(open(p)) for p in glob.glob('results/bottleneck/*.json')]
print(len(rows), 'testbed results')
try:                                         # never let the summary stop the notebook
    df = pd.DataFrame(rows)[['name', 'kind', 'tokens', 'seed', 'gIoU', 'cIoU', 'minutes']].sort_values(['tokens', 'name'])
    print(df.to_string(index=False))
    print(df.assign(cfg=df.name.str.replace(r'_s\d$', '', regex=True)).groupby('cfg')[['gIoU', 'cIoU']].agg(['mean', 'std']).round(4))
except Exception as e:
    print('summary failed:', repr(e))
!grep -h FAILED -A40 results/logs/*lane*.log | head -120

In [ ]:
!cd /kaggle/working && zip -qr bac_testbed_results.zip results/bottleneck results/logs && ls -la bac_testbed_results.zip
assert rows, 'NO RESULTS - see the FAILED blocks above and results/logs/'